In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.optimize import minimize

In [ ]:
source = r'/Users/florenciafuentes/Desktop'


In [ ]:
pf = pd.read_csv(source + '/flo022.csv', usecols = [0,2] , names = ['freq','dbm'] , header = 2)
pf_tr1a  = pf.iloc[0:600].copy()
# Trace 2: rows 606..1206 (inclusive)
pf_tr2a  = pf.iloc[606:1207].copy()
# grafico 1 documento 1
x1aa = pf_tr1a['freq'].to_numpy()
y1aa = pf_tr1a['dbm'].to_numpy()
# grafico 2 documento 1
x2aa = pf_tr2a['freq'].to_numpy()
y2aa = pf_tr2a['dbm'].to_numpy()
#limpiarlos
f1a = np.genfromtxt(x1aa, delimiter=',') #freq traza 1
d1a =  np.genfromtxt(y1aa, delimiter=',') #dbm traza 1

f2a = np.genfromtxt(x2aa, delimiter=',') #freq traza 2
d2a = np.genfromtxt(y2aa, delimiter=',') #dbm traza 2
#ahora si...

In [ ]:
# funciones de conversion

def dbm_to_watts(dbm):
    return 10.0**((np.asarray(dbm) - 30.0)/10.0)

def watts_to_dbm(Pw):
    Pw = np.asarray(Pw)
    return 10.0*np.log10(np.maximum(Pw, 1e-18)) + 30.0  # avoid log(0)

In [ ]:
max_a = f1a[np.where(d1a==np.max(d1a))] #frecuencia max traza 1
fig, ax = plt.subplots(1,1, figsize =(6,4))

ax.plot(f1a,d1a, lw=2.5, c='teal', ls='--', alpha=0.7, label = 'feedback')
ax.plot(f2a,d2a, lw=2.5, c='steelblue', alpha=0.9, label = 'no feedback')

ax.scatter(max_a,(np.max(d1a)), c='plum')
ax.scatter(f2a[np.where(d2a==np.max(d2a))],np.max(d2a), c='purple' )

# ax[1].plot(f1b,d1b, lw=2.5)
# ax[1].plot(f2b,d2b, lw=2.5)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Signal (dBm)')

# ax[1].set_xlabel('Frequency (Hz)')
# ax[1].set_ylabel('Signal (dBm)')

# Legend
ax.legend()
# loc='upper center', bbox_to_anchor=(0.5, 1.15),
          # ncol=5, fancybox=True, shadow=True)

fig.tight_layout()
plt.show()

In [ ]:
tau = 2.38e-6

In [ ]:
# --- frequency axis ---
f = f1a #freq traza 1
w = 2*np.pi*f # omega # laser frequency

# --- convert y to linear W (you were already fitting in W) ---
P_lin = dbm_to_watts(d1a) # dbm traza 1 en watts

# --- instrument widths from RBW (Gaussian IF 'frecuencia intermedia´) ---
RBW = 100e3 #100kHz
VBW = 3e3 #video filter   # not used by the model
sigma_f   = RBW / (2*np.sqrt(2*np.log(2)))   # Hz 
sigma_res = 2*np.pi * sigma_f                # rad/s  (good initial guess)

# ---------- model pieces ----------
def S_cont(w, P0, tau_c, Omega, tau):
    x_plus  = w + Omega 
    x_minus = w - Omega

    def term(x):
        den  = 1.0 + (x*tau_c)**2
        pref = 0.5 * (P0**2) * tau_c / den
        expo = np.exp(-np.abs(tau)/tau_c)
        # correct trig arguments: x in rad/s, tau in s
        bracket = 1.0 - expo*(np.cos(x*np.abs(tau)) + np.sin(x*np.abs(tau))/(x*tau_c))
        # safe x→0 limit
        lim = 1.0 - expo*(1.0 + np.abs(tau)/tau_c)
        bracket = np.where(np.isfinite(bracket), bracket, lim)
        return pref*bracket

    return term(x_minus) + term(x_plus) 

def gaussian(w, mu, sigma):
    return np.exp(-0.5*((w-mu)/sigma)**2)/(np.sqrt(2*np.pi)*sigma)

# Original MODEL B, but we'll fix P0, Omega, tau and only fit [tau_c, A_g, sigma_g]
def model_B_fixed(w, tau_c, A_g, sigma_g, P0_fix, Omega_fix, tau):
    # minimal positivity clamp (since Nelder–Mead has no bounds)
    tau_c   = max(tau_c,   1e-12)
    A_g     = max(A_g,     0.0) #area under Gaussian
    sigma_g = max(sigma_g, 1e-12) # width of Gaussian
    return S_cont(w, P0_fix, tau_c, Omega_fix, tau) + A_g * gaussian(w, Omega_fix, sigma_g)

# --- fix P0 and Omega from the data (use LINEAR to locate peak) ---
Omega_fix = 2*np.pi * f[np.argmax(P_lin)] 
P0_fix    = np.sqrt(np.max(P_lin))  # heuristic scale (same as your code)

# ---------- objective for Nelder–Mead ----------
# We fit in linear W to match your previous code. If you prefer PSD, divide both data and model by ENBW.
def sse_theta(theta):
    tau_c, A_g, sigma_g = theta
    pred = model_B_fixed(w, tau_c, A_g, sigma_g, P0_fix, Omega_fix, tau)
    # SSE in linear W
    return np.sum((P_lin - pred)**2)

# ---------- initial guess (same spirit as your curve_fit p0) ----------
theta0 = np.array([1e-5, dbm_to_watts(np.max(d1a)), sigma_res])  # [tau_c, A_g, sigma_g]

# ---------- Nelder–Mead minimize ----------
res = minimize(sse_theta, theta0, method='Nelder-Mead',
               options=dict(maxiter=20000, xatol=1e-12, fatol=1e-12, disp=False))

tau_c_nm, A_g_nm, sigma_g_nm = res.x
fit_B_nm = model_B_fixed(w, tau_c_nm, A_g_nm, sigma_g_nm, P0_fix, Omega_fix, tau)

# ---------- plot in dBm (your original display) ----------
plt.figure(figsize=(9,5))
plt.plot(f, d1a, label="data", linewidth=2.5, color='teal')
plt.plot(f, watts_to_dbm(fit_B_nm), label="Gaussian fit (Nelder–Mead, P0/Ω/τ fixed)",
         linewidth=2.5, linestyle='--', color='red')
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power (dBm)")
plt.title("Curva 1 — Nelder–Mead")
plt.grid(True)
plt.legend()
plt.show()

print("Nelder–Mead result:")
print("  tau_c      =", tau_c_nm, "s")
print("  A_g (area) =", A_g_nm,   "W")
print("  sigma_g    =", sigma_g_nm, "rad/s   (~", sigma_g_nm/(2*np.pi), "Hz)")
print("  Omega/2π   =", Omega_fix/(2*np.pi), "Hz   (fixed)")
print("  P0 (lin)   =", P0_fix, "(fixed)")
print("  success:", res.success, "  message:", res.message)


In [ ]:
# Fit 1

tau = 2.38e-6
df1 = f1a
f = f1a
dBm = d1a
w = 2*np.pi*f

RBW = 100e3                      # Hz
k_enbw = 1.06                    # ~Gaussian IF; adjust if your manual says otherwise
ENBW = k_enbw * RBW   

P_W = dbm_to_watts(d1a)
Sdata = P_W / ENBW 

sigma_f   = RBW / (2*np.sqrt(2*np.log(2)))   # Hz
sigma_res = 2*np.pi * sigma_f  

Omega0 = 2 * np.pi * f[np.argmax(P_W)]
P0_fix    = np.sqrt(np.max(Sdata))   
Omega_fix = 2*np.pi * f[np.argmax(P_W)]


In [ ]:
# Fit 1

tau = 2.38e-6
df1 = f1a
f = f1a
dBm = d1a
w = 2*np.pi*f

RBW = 100e3                      # Hz
k_enbw = 1.06                    # ~Gaussian IF; adjust if your manual says otherwise
ENBW = k_enbw * RBW   

P_W = dbm_to_watts(d1a)
Sdata = P_W / ENBW 

sigma_f   = RBW / (2*np.sqrt(2*np.log(2)))   # Hz
sigma_res = 2*np.pi * sigma_f  

Omega0 = 2 * np.pi * f[np.argmax(P_W)]
P0_fix    = np.sqrt(np.max(Sdata))   
Omega_fix = 2*np.pi * f[np.argmax(P_W)]

FIT_IN_DBM = True  # <- flip to False for linear-PSD fitting

def model_psd_fixed(w, tau_c, A_g, sigma_g, B0, B1, G):
    S = G*S_cont(w, P0_fix, tau_c, Omega_fix, tau) + A_g*gaussian(w, Omega_fix, sigma_g)
    return S + (B0 + B1*(w - w.mean()))                # W/Hz

def obj(theta):
    tau_c, A_g, sigma_g, B0, B1, G, C_dB, S_dB = theta
    # positivity clamps for NM
    tau_c   = max(tau_c,   1e-12);  A_g = max(A_g, 0.0);  sigma_g = max(sigma_g, 1e-12)

    if FIT_IN_DBM:
        # model -> W in RBW -> dBm, then allow small offset/slope in dB (display alignment)
        S_fit   = model_psd_fixed(w, tau_c, A_g, sigma_g, B0, B1, G)      # W/Hz
        P_fit_W = S_fit*ENBW                                             # W
        m_dbm   = watts_to_dbm(P_fit_W)
        m_dbm  += C_dB + S_dB*(f - f.mean())                             # tiny dB alignment
        return np.sum((d1a - m_dbm)**2)
    else:
        # linear PSD fitting (physically clean)
        S_fit = model_psd_fixed(w, tau_c, A_g, sigma_g, B0, B1, G)        # W/Hz
        return np.sum((Sdata - S_fit)**2)

theta0 = np.array([
    1e-5, P_W.max(), sigma_res, np.median(Sdata), 0.0, 1.0, 0.0, 0.0   # + C_dB, S_dB
])

res = minimize(obj, theta0, method="Nelder-Mead",
               options=dict(maxiter=30000, xatol=1e-12, fatol=1e-12))
tau_c_nm, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm, C_dB_nm, S_dB_nm = res.x

# plot exactly like the analyzer (no surprises)
S_fit   = model_psd_fixed(w, tau_c_nm, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm)
P_fit_W = S_fit*ENBW
fit_dBm = watts_to_dbm(P_fit_W) + (C_dB_nm + S_dB_nm*(f - f.mean()))

plt.figure(figsize=(9,5))
plt.plot(f, d1a, label="data", lw=2.5, color='teal')
plt.plot(f, fit_dBm, label="NM fit (in dBm)", lw=2.5, ls='--', color='red')
plt.xlabel("Frequency (Hz)"); plt.ylabel("Power (dBm)"); plt.title("Curva 1 — Nelder–Mead (dBm fit)")
plt.grid(True); plt.legend(); plt.show()

In [ ]:
P_W   = dbm_to_watts(d1a)     # W in RBW
Sdata = P_W           # W/Hz
Omega_fix = 2*np.pi * f[np.argmax(d1a)]
P0_fix    = np.sqrt(np.max(Sdata))

In [ ]:
FIT_IN_DBM = True # <- flip to False for linear-PSD fitting

def model_psd_fixed(w, tau_c, A_g, sigma_g, B0, B1, G):
    S = G*S_cont(w, P0_fix, tau_c, Omega_fix, tau) + A_g*gaussian(w, Omega_fix, sigma_g)
    return S + (B0 + B1*(w - w.mean()))                # W/Hz

def obj(theta):
    tau_c, A_g, sigma_g, B0, B1, G, C_dB, S_dB = theta
    # positivity clamps for NM
    tau_c   = max(tau_c,   1e-12);  A_g = max(A_g, 0.0);  sigma_g = max(sigma_g, 1e-12)

    if FIT_IN_DBM:
        # model -> W in RBW -> dBm, then allow small offset/slope in dB (display alignment)
        S_fit   = model_psd_fixed(w, tau_c, A_g, sigma_g, B0, B1, G)      # W/Hz
        P_fit_W = S_fit*ENBW                                             # W
        m_dbm   = watts_to_dbm(P_fit_W)
        m_dbm  += C_dB + S_dB*(f - f.mean())                             # tiny dB alignment
        return np.sum((d1a - m_dbm)**2)
    else:
        # linear PSD fitting (physically clean)
        S_fit = model_psd_fixed(w, tau_c, A_g, sigma_g, B0, B1, G)        # W/Hz
        return np.sum((Sdata - S_fit)**2)

theta0 = np.array([
    1e-5, P_W.max(), sigma_res, np.median(Sdata), 0.0, 1.0, 0.0, 0.0   # + C_dB, S_dB
])

res = minimize(obj, theta0, method="Nelder-Mead",
               options=dict(maxiter=30000, xatol=1e-12, fatol=1e-12))
tau_c_nm, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm, C_dB_nm, S_dB_nm = res.x

# plot exactly like the analyzer (no surprises)
S_fit   = model_psd_fixed(w, tau_c_nm, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm)
P_fit_W = S_fit*ENBW
fit_dBm = watts_to_dbm(P_fit_W) + (C_dB_nm + S_dB_nm*(f - f.mean()))

In [ ]:
# --- choose a trace ---
tau = 2.38e-6

df1 = f1a
f1 = df1
df2 = f2a
f2 = df2

w1 = 2*np.pi*f1                    # rad/s
w2 = 2*np.pi*f2                   # rad/s

dBm1 = d1a                        # analyzer trace (power in RBW)
dBm2 = d2a

RBW = 100e3                      # Hz
k_enbw = 1.06                    # ~Gaussian IF; adjust if your manual says otherwise
ENBW = k_enbw * RBW   

# Convert y (dBm) to linear power (W). Fit in linear space for stability.
PWa = dbm_to_watts(d1a)
PWb = dbm_to_watts(d2a)

Sdata1 = PWa / ENBW 
Sdata2 = PWb / ENBW 

sigma_f   = RBW / (2*np.sqrt(2*np.log(2)))   # Hz
sigma_res = 2*np.pi * sigma_f  

# ---- initial guesses (tweak if needed) ----
# Omega guess from where the peak is (in rad/s):
Omega01 = 2 * np.pi * f[np.argmax(PWa)]
Omega02 = 2 * np.pi * f[np.argmax(PWb)]

P0_fixa    = np.sqrt(np.max(Sdata1))        # scale heuristic for S_cont
P0_fixb   = np.sqrt(np.max(Sdata2))        # scale heuristic for S_cont

thetaa = np.array([ 1e-6, PWa.max(), sigma_res, np.median(Sdata1), 0.0, 1.0, 0.0, 0.0 ])
thetab = np.array([ 1e-6, PWb.max(), sigma_res, np.median(Sdata2), 0.0, 1.0, 0.0, 0.0 ])

res1 = minimize(obj, thetaa, method="Nelder-Mead",
               options=dict(maxiter=30000, xatol=1e-12, fatol=1e-12))
tau_c_nma, A_g_nma, sigma_g_nma, B0_nma, B1_nma, G_nma, C_dB_nma, S_dB_nma = res1.x

res2 = minimize(obj, thetab, method="Nelder-Mead",
               options=dict(maxiter=30000, xatol=1e-12, fatol=1e-12))
tau_c_nmb, A_g_nmb, sigma_g_nmb, B0_nmb, B1_nmb, G_nmb, C_dB_nmb, S_dB_nmb = res2.x

# plot exactly like the analyzer (no surprises)
S_fit1   = model_psd_fixed(w1, tau_c_nma, A_g_nma, sigma_g_nma, B0_nma, B1_nma, G_nma)
S_fit2   = model_psd_fixed(w2, tau_c_nmb, A_g_nmb, sigma_g_nmb, B0_nmb, B1_nmb, G_nmb)

P_fit_W1 = S_fit1*ENBW
P_fit_W2 = S_fit2*ENBW

fit_dBm1 = watts_to_dbm(P_fit_W1) + (C_dB_nma + S_dB_nma*(f1 - f1.mean()))
fit_dBm2 = watts_to_dbm(P_fit_W2) + (C_dB_nmb + S_dB_nmb*(f2 - f2.mean()))
plt.figure(figsize=(9,5))
plt.plot(f, d1a, label="data", lw=2.5, color='teal')
plt.plot(f, fit_dBm, label="NM fit (in dBm)", lw=2.5, ls='--', color='red')
plt.xlabel("Frequency (Hz)"); plt.ylabel("Power (dBm)"); plt.title("Curva 1 — Nelder–Mead (dBm fit)")
plt.grid(True); plt.legend(); plt.show()

In [ ]:
P_W   = dbm_to_watts(d2a)     # W in RBW
Sdata = P_W           # W/Hz
Omega_fix = 2*np.pi * f2[np.argmax(d2a)]
P0_fix    = np.sqrt(np.max(Sdata))

In [ ]:
FIT_IN_DBM = True  # <- flip to False for linear-PSD fitting

def model_psd_fixed2(w, tau_c, A_g, sigma_g, B0, B1, G):
    S = G*S_cont(w2, P0_fix, tau_c, Omega_fix, tau) + A_g*gaussian(w2, Omega_fix, sigma_g)
    return S + (B0 + B1*(w - w.mean()))                # W/Hz

def obj2(theta):
    tau_c, A_g, sigma_g, B0, B1, G, C_dB, S_dB = theta
    # positivity clamps for NM
    tau_c   = max(tau_c,   1e-12);  A_g = max(A_g, 0.0);  sigma_g = max(sigma_g, 1e-12)

    if FIT_IN_DBM:
        # model -> W in RBW -> dBm, then allow small offset/slope in dB (display alignment)
        S_fit   = model_psd_fixed2(w2, tau_c, A_g, sigma_g, B0, B1, G)      # W/Hz
        P_fit_W = S_fit*ENBW                                             # W
        m_dbm   = watts_to_dbm(P_fit_W)
        m_dbm  += C_dB + S_dB*(f2 - f2.mean())                             # tiny dB alignment
        return np.sum((d2a - m_dbm)**2)
    else:
        # linear PSD fitting (physically clean)
        S_fit = model_psd_fixed2(w2, tau_c, A_g, sigma_g, B0, B1, G)        # W/Hz
        return np.sum((Sdata - S_fit)**2)

theta0 = np.array([
    1e-7, P_W.max(), sigma_res, np.median(Sdata), 0.0, 1.0, 0.0, 0.0   # + C_dB, S_dB
])

res = minimize(obj2, theta0, method="Nelder-Mead",
               options=dict(maxiter=30000, xatol=1e-12, fatol=1e-12))
tau_c_nmb, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm, C_dB_nm, S_dB_nm = res.x

# plot exactly like the analyzer (no surprises)
S_fit   = model_psd_fixed2(w2, tau_c_nmb, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm)
P_fit_W = S_fit*ENBW
fit_dBm2 = watts_to_dbm(P_fit_W) + (C_dB_nm + S_dB_nm*(f2 - f2.mean()))
print(tau_c_nm)
print(tau_c_nmb)

In [ ]:
#plot -------------------------------------------------------------------------
fig, axes = plt.subplots(1,2, figsize =(10,4))

axes[0].plot(f1a*1e-6, d1a, lw=3, c='teal', alpha=0.8, label = 'feedback')
axes[0].plot(f1a*1e-6, fit_dBm1, label="fit", linewidth=1.5, linestyle = '-', alpha=0.6, color = 'red')
axes[0].plot( 20,-20,label = r'$\tau_c = $'+"{0:.5g}".format(tau_c_nm))

axes[1].plot(f2a*1e-6, d2a, lw=3,  c='teal', alpha=0.8, label = 'no feedback')
axes[1].plot(f2a*1e-6, fit_dBm2, label="fit", linewidth=1.5, linestyle = '-', alpha=0.6, color = 'red')
axes[1].plot( 20,-20,label = r'$\tau_c = $'+"{0:.5g}".format(tau_c_nmb))

axes[0].set_xlabel('Frequency (MHz)')
axes[0].set_ylabel('Signal (dBm)')

axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Signal (dBm)')

# Legend
axes[0].legend()
axes[1].legend()


fig.tight_layout()
plt.show()

In [ ]:
print('feedback=',1./(np.pi * tau_c_nm)*1e-6 ,' MHz')
print('no feedback=',1./(np.pi * tau_c_nmb)*1e-6 ,' MHz')